# 🔑 Solutions — go or nogo? — the real neuron, on your own

One worked answer per exercise. There is usually more than one correct
way to write these — if your self-check printed ✅, yours is right too.

Read this *after* an attempt: seeing the answer teaches far less than
getting stuck did.

In [ ]:
# Run this first. It loads two neurons' recordings from the lecture —
# from data/ if you downloaded them, otherwise straight from GitHub.
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab05/data/"
DATA = Path("data")            # same layout as a uv-start project
DATA.mkdir(exist_ok=True)
NEURON_1 = DATA / "neuron_0588_20180828_5_lecture.csv"
NEURON_2 = DATA / "neuron_0406_20180620_8_lecture.csv"
for path in [NEURON_1, NEURON_2]:
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL + path.name, path)

# Columns: 0 frame, 1 dff, 2 go, 3 nogo, 4 reward, 5 lick
# unpack=True hands back each column as its own array - three loads in one line
dff, go, nogo = np.loadtxt(NEURON_1, delimiter=",", skiprows=1, usecols=(1, 2, 3), unpack=True)

FRAME_RATE = 10.8   # frames per second
print("neuron 1:", dff.size, "frames")

## Solution — Checkpoint 1: `go_starts` and `nogo_starts`

In [ ]:
go_starts = np.where(go == 1)[0][::9]
nogo_starts = np.where(nogo == 1)[0][::9]
print(len(go_starts), "go trials,", len(nogo_starts), "nogo trials")

## Solution — Checkpoint 2: `go_trials` and `nogo_trials`

In [ ]:
def stack_trials(trace, starts):
    trials = []
    for start in starts:
        if start >= 10:                   # the first nogo trial starts at frame 2
            trials.append(trace[start - 10 : start + 30])
    return np.array(trials)

go_trials = stack_trials(dff, go_starts)
nogo_trials = stack_trials(dff, nogo_starts)
print(go_trials.shape, nogo_trials.shape)

## Solution — Checkpoint 3: the average trial

In [ ]:
go_response = go_trials.mean(axis=0)
nogo_response = nogo_trials.mean(axis=0)
go_peak = round(go_response[10:].max(), 3)
nogo_peak = round(nogo_response[10:].max(), 3)
print("go peak", go_peak, "| nogo peak", nogo_peak)

## Solution — Checkpoint 4: `response_to`, and a second neuron

In [ ]:
def response_to(trace, marker):
    """The average trial round every start of a 0/1 marker."""
    starts = np.where(marker == 1)[0][::9]
    return stack_trials(trace, starts).mean(axis=0)

dff_2, go_2, nogo_2 = np.loadtxt(NEURON_2, delimiter=",", skiprows=1, usecols=(1, 2, 3), unpack=True)

ratio_1 = round(response_to(dff, go)[10:].max() / response_to(dff, nogo)[10:].max(), 1)
ratio_2 = round(response_to(dff_2, go_2)[10:].max() / response_to(dff_2, nogo_2)[10:].max(), 1)
print("neuron 1: go is", ratio_1, "times nogo")
print("neuron 2: go is", ratio_2, "times nogo")

In [ ]:
# Run me - Part 1's way of doing it, kept as reference functions
def starts_by_nine(marker):
    """The first frame of each trial, assuming every word is exactly 9 frames."""
    return np.where(marker == 1)[0][::9]

def trials_by_loop(trace, starts):
    """A (trials x 40) grid: 10 frames before each start, 30 after."""
    trials = []
    for i in starts:
        if i >= 10:
            trials.append(trace[i - 10 : i + 30])
    return np.array(trials)

print(trials_by_loop(dff, starts_by_nine(go)).shape)

## Solution — Stretch 1, 🐞 broken: one frame early

In [ ]:
go_starts = np.where(np.diff(go) == 1)[0] + 1     # diff[i] compares frame i+1 with frame i
print(go_starts[:5])

## Solution — Stretch 1, 🔬 write it: test the assumption

In [ ]:
go_ends = np.where(np.diff(go) == -1)[0]       # the last frame that is still on
durations = go_ends - go_starts + 1            # element by element: one duration per trial
all_nine = (durations == 9).all()
print(durations[:10], "-> all nine?", all_nine)

In [ ]:
# DEMO - a small grid first, then all 254 go trials at once
starts = np.array([43, 125, 166])
grid = starts[:, np.newaxis] + np.arange(-2, 3)
print(grid)
print(dff[grid].round(2))           # the values at those positions, same shape

offsets = np.arange(-10, 30)        # 10 before, 30 after - the window from Part 1
go_trials = dff[go_starts[:, np.newaxis] + offsets]
print(go_trials.shape)
print("same as the loop?", np.array_equal(go_trials, trials_by_loop(dff, go_starts)))

## Solution — Stretch 2, 🔬 write it: nogo, without a loop

In [ ]:
nogo_starts = np.where(np.diff(nogo) == 1)[0] + 1
nogo_ok = nogo_starts[nogo_starts >= 10]          # boolean filtering: drop the trial at frame 2
nogo_trials = dff[nogo_ok[:, np.newaxis] + offsets]
print(nogo_trials.shape)

## Solution — Stretch 3, 🐞 broken: the baseline that won't subtract

In [ ]:
baseline = go_trials[:, :10].mean(axis=1)
go_corrected = go_trials - baseline[:, np.newaxis]   # (254, 40) - (254, 1): one baseline per row
print(go_corrected.shape)

## Solution — Stretch 4, 🔬 write it: one score per trial

In [ ]:
nogo_corrected = nogo_trials - nogo_trials[:, :10].mean(axis=1, keepdims=True)
go_scores = go_corrected[:, 10:].mean(axis=1)        # along each row: one score per trial
nogo_scores = nogo_corrected[:, 10:].mean(axis=1)
print("go:  ", go_scores.shape, round(go_scores.mean(), 3))
print("nogo:", nogo_scores.shape, round(nogo_scores.mean(), 3))

In [ ]:
# DEMO - the observed difference, and ONE shuffle
observed = go_scores.mean() - nogo_scores.mean()
print("observed difference:", round(observed, 3))

pooled = np.concatenate([go_scores, nogo_scores])    # all 517 scores in one array
n_go = len(go_scores)

rng = np.random.default_rng(0)
shuffled = rng.permutation(pooled)
fake_go, fake_nogo = shuffled[:n_go], shuffled[n_go:]
print("one shuffled difference:", round(fake_go.mean() - fake_nogo.mean(), 3))

## Solution — Stretch 5, 🔬 write it: 1,000 shuffles

In [ ]:
fake_diffs = []
for _ in range(1000):
    shuffled = rng.permutation(pooled)
    fake_diffs.append(shuffled[:n_go].mean() - shuffled[n_go:].mean())
fake_diffs = np.array(fake_diffs)

p_value = (fake_diffs >= observed).mean()      # the fraction of shuffles at least as big
print("biggest shuffled difference:", round(fake_diffs.max(), 3))
print("p =", p_value)

## Solution — 🏆 Finale: `shuffle_test`

In [ ]:
def scores(trace, marker):
    """One baseline-corrected score per trial."""
    starts = np.where(np.diff(marker) == 1)[0] + 1
    starts = starts[starts >= 10]
    trials = trace[starts[:, np.newaxis] + np.arange(-10, 30)]
    corrected = trials - trials[:, :10].mean(axis=1, keepdims=True)
    return corrected[:, 10:].mean(axis=1)

def shuffle_test(trace, go_marker, nogo_marker):
    """The go-minus-nogo difference, and the fraction of 1000 shuffles that reach it."""
    go_s = scores(trace, go_marker)
    nogo_s = scores(trace, nogo_marker)
    observed = go_s.mean() - nogo_s.mean()
    pooled = np.concatenate([go_s, nogo_s])
    rng = np.random.default_rng(0)
    fake = []
    for _ in range(1000):
        shuffled = rng.permutation(pooled)
        fake.append(shuffled[:len(go_s)].mean() - shuffled[len(go_s):].mean())
    return observed, (np.array(fake) >= observed).mean()

for name, (trace, g, n) in {"neuron 1": (dff, go, nogo), "neuron 2": (dff_2, go_2, nogo_2)}.items():
    diff, p = shuffle_test(trace, g, n)
    print(f"{name}: difference {diff:+.3f}, p = {p:.3f}")